In [3]:
!pip install autogluon.tabular ucimlrepo google-adk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 13.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.4/304.4 kB 13.4 MB/s eta 0:00:00


In [9]:
import os
os.environ["RAY_DISABLE_METRICS_COLLECTION"] = "1"
os.environ["RAY_ENABLE_MAC"] = "0"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix, f1_score
from ucimlrepo import fetch_ucirepo
from autogluon.tabular import TabularDataset, TabularPredictor

# 1. Fetch Dataset
print("Fetching dataset...")
mi_data = fetch_ucirepo(id=579)
X_full = mi_data.data.features.copy()
y_full = mi_data.data.targets.copy()

y_full = y_full.fillna(y_full.mode().iloc[0])

# ALTERAÇÃO: Limitado para o teste do 1º alvo apenas
target_names = [y_full.columns.tolist()[0]] 
binary_targets = target_names

# 2. Temporal Masking (Admission)
def get_admission_data(X_data):
    day_1_cols = ['R_AB_1_n', 'NA_R_1_n', 'NOT_NA_1_n']
    day_2_cols = ['R_AB_2_n', 'NA_R_2_n', 'NOT_NA_2_n']
    day_3_cols = ['R_AB_3_n', 'NA_R_3_n', 'NOT_NA_3_n']
    drop_cols = day_1_cols + day_2_cols + day_3_cols
    return X_data.drop(columns=[c for c in drop_cols if c in X_data.columns])

X_adm = get_admission_data(X_full)
df_full = pd.concat([X_adm, y_full], axis=1)

# 3. Train/Test Split
train_data, test_data = train_test_split(df_full, test_size=0.2, random_state=42)
train_data = TabularDataset(train_data)
test_data = TabularDataset(test_data)

predictors = {}
predicted_train_features = train_data.copy()
predicted_test_features = test_data.copy()

print(f"\nTraining AutoGluon for Single Target: {target_names[0]}...")

# PENALIDADE: Multiplicador de agressividade (5x) para a classe rara
PENALTY_FACTOR = 5.0 

target = target_names[0]

# --- INTERVENÇÃO 1: SAMPLE WEIGHTS COM HIPER-PENALIDADE ---
num_pos = (predicted_train_features[target] == 1).sum()
num_neg = (predicted_train_features[target] == 0).sum()
weight_ratio = (num_neg / (num_pos + 1e-5)) * PENALTY_FACTOR 

predicted_train_features['sample_weight'] = np.where(predicted_train_features[target] == 1, weight_ratio, 1.0)
weight_col = 'sample_weight'

# --- INTERVENÇÃO 2: TREINAMENTO OTIMIZADO DE ALTA PERFORMANCE ---
predictor = TabularPredictor(
    label=target, 
    eval_metric='roc_auc',
    problem_type='binary',
    sample_weight=weight_col
).fit(
    predicted_train_features, 
    presets='good_quality', 
    time_limit=180, 
    excluded_model_types=['KNN', 'RF', 'XT'], # Remove modelos fracos/lentos
    auto_stack=True, # Ativa Stacking para máxima qualidade
    num_gpus=1, # Se houver GPU, força o uso para acelerar LightGBM e XGBoost
    verbosity=0 
)

predictors[target] = predictor

Fetching dataset...


No path specified. Models will be saved in: "AutogluonModels/ag-20260323_134534"



Training AutoGluon for Single Target: FIBR_PREDS...

OTIMIZAÇÃO DE LIMIAR (THRESHOLD SWEEP)

--- Análise de Limiar para: FIBR_PREDS ---
Limiar     | Recall     | Acurácia   | F1-Score  
--------------------------------------------------


NameError: name 'f1_score' is not defined

In [12]:
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix, f1_score, precision_score
import numpy as np

# Configurações iniciais
thresholds_to_test = np.arange(0.05, 0.95, 0.05)
MIN_ACCURACY_ACCEPTABLE = 0.60 

y_true = test_data[target].values
y_probs = predictors[target].predict_proba(test_input).iloc[:, 1].values

print(f"\n{'Limiar':<7} | {'Recall':<7} | {'Specif':<7} | {'Prec':<7} | {'Acc':<7} | {'F1':<7} | {'Neg(P)':<7} | {'Pos(P)':<7}")
print("-" * 85)

best_threshold = 0.50
best_recall = 0.0

for thresh in thresholds_to_test:
    y_pred_thresh = (y_probs >= thresh).astype(int)
    
    # Cálculo de Matriz de Confusão para métricas manuais
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_thresh).ravel()
    
    # Métricas
    rec  = recall_score(y_true, y_pred_thresh, zero_division=0) # Mesma coisa que Sensitivity
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    prec = precision_score(y_true, y_pred_thresh, zero_division=0)
    acc  = accuracy_score(y_true, y_pred_thresh)
    f1   = f1_score(y_true, y_pred_thresh, zero_division=0)
    
    # Contagem de predições
    n_pos = np.sum(y_pred_thresh)
    n_neg = len(y_pred_thresh) - n_pos

    print(f"{thresh:<7.2f} | {rec:<7.3f} | {spec:<7.3f} | {prec:<7.3f} | {acc:<7.3f} | {f1:<7.3f} | {n_neg:<7} | {n_pos:<7}")
    
    # Lógica de decisão (Exemplo: Priorizando Recall mantendo Acurácia mínima)
    if rec >= best_recall and acc >= MIN_ACCURACY_ACCEPTABLE:
        best_recall = rec
        best_threshold = thresh
        best_acc = acc

print("-" * 85)
print(f"Sugestão: Limiar {best_threshold:.2f} (Recall: {best_recall:.4f}, Acc: {best_acc:.4f})")


Limiar  | Recall  | Specif  | Prec    | Acc     | F1      | Neg(P)  | Pos(P) 
-------------------------------------------------------------------------------------
0.05    | 1.000   | 0.000   | 0.103   | 0.103   | 0.187   | 0       | 340    
0.10    | 1.000   | 0.000   | 0.103   | 0.103   | 0.187   | 0       | 340    
0.15    | 1.000   | 0.000   | 0.103   | 0.103   | 0.187   | 0       | 340    
0.20    | 1.000   | 0.000   | 0.103   | 0.103   | 0.187   | 0       | 340    
0.25    | 1.000   | 0.000   | 0.103   | 0.103   | 0.187   | 0       | 340    
0.30    | 1.000   | 0.000   | 0.103   | 0.103   | 0.187   | 0       | 340    
0.35    | 1.000   | 0.016   | 0.104   | 0.118   | 0.189   | 5       | 335    
0.40    | 1.000   | 0.030   | 0.106   | 0.129   | 0.191   | 9       | 331    
0.45    | 1.000   | 0.059   | 0.109   | 0.156   | 0.196   | 18      | 322    
0.50    | 1.000   | 0.102   | 0.113   | 0.194   | 0.203   | 31      | 309    
0.55    | 0.971   | 0.190   | 0.121   | 0.271   | 0.215

In [13]:
import os
os.environ["RAY_DISABLE_METRICS_COLLECTION"] = "1"
os.environ["RAY_ENABLE_MAC"] = "0"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix, f1_score
from ucimlrepo import fetch_ucirepo
from autogluon.tabular import TabularDataset, TabularPredictor

print("Fetching dataset...")
mi_data = fetch_ucirepo(id=579)
X_full = mi_data.data.features.copy()
y_full = mi_data.data.targets.copy()

y_full = y_full.fillna(y_full.mode().iloc[0])

# Retornando para os 12 alvos
target_names = y_full.columns.tolist()
binary_targets = target_names[:-1]
multiclass_target = target_names[-1]

def get_admission_data(X_data):
    day_1_cols = ['R_AB_1_n', 'NA_R_1_n', 'NOT_NA_1_n']
    day_2_cols = ['R_AB_2_n', 'NA_R_2_n', 'NOT_NA_2_n']
    day_3_cols = ['R_AB_3_n', 'NA_R_3_n', 'NOT_NA_3_n']
    drop_cols = day_1_cols + day_2_cols + day_3_cols
    return X_data.drop(columns=[c for c in drop_cols if c in X_data.columns])

X_adm = get_admission_data(X_full)
df_full = pd.concat([X_adm, y_full], axis=1)

train_data, test_data = train_test_split(df_full, test_size=0.2, random_state=42)
train_data = TabularDataset(train_data)
test_data = TabularDataset(test_data)

predictors = {}
predicted_train_features = train_data.copy()
predicted_test_features = test_data.copy()

print("\nTraining Classifier Chain for ALL 12 Targets...")

PENALTY_FACTOR = 5.0 

for i, target in enumerate(target_names):
    print(f"\n--- Training Model for Target: {target} ({i+1}/{len(target_names)}) ---")
    
    future_targets = target_names[i+1:]
    train_input = predicted_train_features.drop(columns=future_targets).copy()
    
    if target in binary_targets:
        num_pos = (train_input[target] == 1).sum()
        num_neg = (train_input[target] == 0).sum()
        weight_ratio = (num_neg / (num_pos + 1e-5)) * PENALTY_FACTOR
        
        train_input['sample_weight'] = np.where(train_input[target] == 1, weight_ratio, 1.0)
        weight_col = 'sample_weight'
        metric = 'roc_auc'
        prob_type = 'binary'
    else:
        weight_col = None
        metric = 'accuracy'
        prob_type = 'multiclass'
        
predictor = TabularPredictor(
        label=target, 
        eval_metric=metric,
        problem_type=prob_type,
        sample_weight=weight_col
    ).fit(
        train_input, 
        presets='best_quality',
        time_limit=600,
        excluded_model_types=['NN_TORCH', 'FASTAI'], 
        verbosity=0 
    )
    
    predictors[target] = predictor
    
    if weight_col:
        train_input = train_input.drop(columns=[weight_col])
        
    predicted_train_features[target] = predictor.predict(train_input)
    
    test_input = predicted_test_features.drop(columns=future_targets)
    predicted_test_features[target] = predictor.predict(test_input)

Fetching dataset...


No path specified. Models will be saved in: "AutogluonModels/ag-20260323_135616"



Training Classifier Chain for ALL 12 Targets...

--- Training Model for Target: FIBR_PREDS (1/12) ---


No path specified. Models will be saved in: "AutogluonModels/ag-20260323_135930"



--- Training Model for Target: PREDS_TAH (2/12) ---


...


--- Training Model for Target: REC_IM (10/12) ---


No path specified. Models will be saved in: "AutogluonModels/ag-20260323_142812"



--- Training Model for Target: P_IM_STEN (11/12) ---


No path specified. Models will be saved in: "AutogluonModels/ag-20260323_143125"



--- Training Model for Target: LET_IS (12/12) ---

AVALIAÇÃO DE CADEIA COM OTIMIZAÇÃO DE LIMIAR (Foco em Recall)

--- Resultados para: FIBR_PREDS ---
Limiar Ideal: 0.65
Recall: 0.8286 | Acurácia: 0.3676 | F1: 0.2125
Matriz de Confusão:
[[ 96 209]
 [  6  29]]

--- Resultados para: PREDS_TAH ---
Limiar Ideal: 0.65
Recall: 1.0000 | Acurácia: 0.1588 | F1: 0.0205
Matriz de Confusão:
[[ 51 286]
 [  0   3]]

--- Resultados para: JELUD_TAH ---
Limiar Ideal: 0.45
Recall: 1.0000 | Acurácia: 0.2265 | F1: 0.0295
Matriz de Confusão:
[[ 73 263]
 [  0   4]]

--- Resultados para: FIBR_JELUD ---
Limiar Ideal: 0.50
Recall: 0.8462 | Acurácia: 0.2853 | F1: 0.0830
Matriz de Confusão:
[[ 86 241]
 [  2  11]]

--- Resultados para: A_V_BLOK ---
Limiar Ideal: 0.50
Recall: 0.8182 | Acurácia: 0.7588 | F1: 0.1800
Matriz de Confusão:
[[249  80]
 [  2   9]]

--- Resultados para: OTEK_LANC ---
Limiar Ideal: 0.50
Recall: 0.9565 | Acurácia: 0.3324 | F1: 0.1624
Matriz de Confusão:
[[ 91 226]
 [  1  22]]

--- Resultados

In [24]:
from sklearn.metrics import fbeta_score

print("\n" + "="*60)
print("AVALIAÇÃO DE CADEIA COM OTIMIZAÇÃO DE LIMIAR (Foco em Recall)")
print("="*60)

thresholds_to_test = np.arange(0.4, 0.8, 0.005)

# NOVA REGRA DE NEGÓCIO: O Recall deve ser de pelo menos 70%
MIN_RECALL_ACCEPTABLE = 0.8

for i, target in enumerate(target_names):
    y_true = test_data[target].values
    future_targets = target_names[i+1:]
    test_input = predicted_test_features.drop(columns=future_targets)
    
    if target in binary_targets:
        print(f"\n--- Resultados para: {target} ---")
        y_probs = predictors[target].predict_proba(test_input).iloc[:, 1].values
        
        best_threshold = 0.50
        best_recall = 0.0
        best_acc = 0.0
        
        for thresh in thresholds_to_test:
            y_pred_thresh = (y_probs >= thresh).astype(int)
            rec = recall_score(y_true, y_pred_thresh, zero_division=0)
            acc = accuracy_score(y_true, y_pred_thresh)
            
            # Buscamos a maior acurácia possível, desde que o recall não caia abaixo de 70%
            if rec >= MIN_RECALL_ACCEPTABLE and acc > best_acc:
                best_acc = acc
                best_threshold = thresh
                best_recall = rec
                
        # Calcula a matriz final para o limiar escolhido
        y_pred_final = (y_probs >= best_threshold).astype(int)
        cm = confusion_matrix(y_true, y_pred_final)
        f1 = fbeta_score(y_true, y_pred_final, beta=3, zero_division=0)
                
        print(f"Limiar Ideal: {best_threshold:.3f}")
        print(f"Recall: {best_recall:.4f} | Acurácia: {best_acc:.4f} | F1: {f1:.4f}")
        print(f"Matriz de Confusão:\n{cm}")

    else:
        print(f"\n--- Resultados para: {target} (Multiclasse) ---")
        y_pred = predictors[target].predict(test_input).values
        acc = accuracy_score(y_true, y_pred)
        rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        cm = confusion_matrix(y_true, y_pred)
        
        print(f"Acurácia: {acc:.4f} | Weighted Recall: {rec:.4f}")
        print(f"Matriz de Confusão:\n{cm}")


AVALIAÇÃO DE CADEIA COM OTIMIZAÇÃO DE LIMIAR (Foco em Recall)

--- Resultados para: FIBR_PREDS ---
Limiar Ideal: 0.675
Recall: 0.8000 | Acurácia: 0.4441 | F1: 0.5333
Matriz de Confusão:
[[123 182]
 [  7  28]]

--- Resultados para: PREDS_TAH ---
Limiar Ideal: 0.670
Recall: 1.0000 | Acurácia: 0.3912 | F1: 0.1266
Matriz de Confusão:
[[130 207]
 [  0   3]]

--- Resultados para: JELUD_TAH ---
Limiar Ideal: 0.495
Recall: 1.0000 | Acurácia: 0.6853 | F1: 0.2721
Matriz de Confusão:
[[229 107]
 [  0   4]]

--- Resultados para: FIBR_JELUD ---
Limiar Ideal: 0.515
Recall: 0.8462 | Acurácia: 0.3794 | F1: 0.3264
Matriz de Confusão:
[[118 209]
 [  2  11]]

--- Resultados para: A_V_BLOK ---
Limiar Ideal: 0.525
Recall: 0.8182 | Acurácia: 0.8676 | F1: 0.5960
Matriz de Confusão:
[[286  43]
 [  2   9]]

--- Resultados para: OTEK_LANC ---
Limiar Ideal: 0.525
Recall: 0.8261 | Acurácia: 0.4382 | F1: 0.4600
Matriz de Confusão:
[[130 187]
 [  4  19]]

--- Resultados para: RAZRIV ---
Limiar Ideal: 0.645
Recall:

In [25]:
import pandas as pd

print("\n" + "="*60)
print("INVESTIGAÇÃO DE IMPORTÂNCIA DE VARIÁVEIS (TOP 10)")
print("="*60)

# Focaremos na análise das duas complicações com melhor performance de triagem
targets_de_interesse = ['A_V_BLOK', 'RAZRIV']

for target in targets_de_interesse:
    print(f"\n--- Extraindo importância para: {target} ---")
    
    # 1. Reconstruir o ambiente exato que o modelo viu durante a predição
    # Como usamos Classifier Chain, o modelo viu as features originais + predições dos targets anteriores
    target_index = target_names.index(target)
    future_targets = target_names[target_index+1:]
    
    data_for_importance = predicted_test_features.drop(columns=future_targets)
    
    # 2. Executar a Permutação de Importância no AutoGluon
    # O AutoGluon calcula isso automaticamente sobre o dado fornecido
    # Nota: Este processo embaralha as colunas iterativamente, pode levar alguns segundos.
    importance_df = predictors[target].feature_importance(data_for_importance)
    
    # 3. Formatação da Saída
    # Exibe apenas as 10 variáveis com maior impacto na queda de performance (importance)
    print(importance_df[['importance', 'stddev', 'p_value']].head(10))


INVESTIGAÇÃO DE IMPORTÂNCIA DE VARIÁVEIS (TOP 10)

--- Extraindo importância para: A_V_BLOK ---
            importance    stddev   p_value
inf_im        0.224567  0.026615  0.000023
ROE           0.027284  0.005064  0.000136
O_L_POST      0.021210  0.008009  0.002037
TIME_B_S      0.018380  0.004846  0.000530
ANT_CA_S_n    0.017163  0.005031  0.000793
L_BLOOD       0.016652  0.002894  0.000105
zab_leg_02    0.015032  0.001015  0.000002
K_BLOOD       0.013564  0.003393  0.000433
lat_im        0.013358  0.002303  0.000102
AGE           0.013286  0.002150  0.000080

--- Extraindo importância para: RAZRIV ---
           importance    stddev   p_value
AGE          0.211182  0.016829  0.000005
S_AD_ORIT    0.058909  0.007971  0.000039
DLIT_AG      0.037973  0.002432  0.000002
TIME_B_S     0.018539  0.002665  0.000050
ant_im       0.010795  0.001627  0.000060
AST_BLOOD    0.010417  0.003250  0.001003
GB           0.005551  0.001395  0.000441
NA_BLOOD     0.004662  0.000897  0.000156
L_BLOOD 

In [ ]:
import os
# Disable Ray metrics agents that cause RpcError in notebooks/containers
os.environ["RAY_DISABLE_METRICS_COLLECTION"] = "1"
os.environ["RAY_ENABLE_MAC"] = "0"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo
from autogluon.tabular import TabularDataset, TabularPredictor

# 1. Fetch Dataset
print("Fetching dataset...")
mi_data = fetch_ucirepo(id=579)
X_full = mi_data.data.features.copy()
y_full = mi_data.data.targets.copy()

# Fill missing targets with the mode
y_full = y_full.fillna(y_full.mode().iloc[0])

target_names = y_full.columns.tolist()
binary_targets = target_names[:-1]
multiclass_target = target_names[-1]

# 2. Base Train/Test Split (Ensures the same patients are evaluated across all days)
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)

# 3. Temporal Datasets Generator
def generate_temporal_datasets(X_data):
    """
    Generates 4 distinct datasets masking future columns to prevent Data Leakage.
    """
    day_1_cols = ['R_AB_1_n', 'NA_R_1_n', 'NOT_NA_1_n']
    day_2_cols = ['R_AB_2_n', 'NA_R_2_n', 'NOT_NA_2_n']
    day_3_cols = ['R_AB_3_n', 'NA_R_3_n', 'NOT_NA_3_n']
    
    datasets = {}
    
    # Admission
    drop_adm = day_1_cols + day_2_cols + day_3_cols
    datasets['admission'] = X_data.drop(columns=[c for c in drop_adm if c in X_data.columns])
    
    # Day 1
    #drop_day_1 = day_2_cols + day_3_cols
    #datasets['day_1'] = X_data.drop(columns=[c for c in drop_day_1 if c in X_data.columns])
    
    # Day 2
    #drop_day_2 = day_3_cols
    #datasets['day_2'] = X_data.drop(columns=[c for c in drop_day_2 if c in X_data.columns])
    
    # Day 3 (All available features)
    #datasets['day_3'] = X_data.copy()
    
    return datasets

# Apply temporal masking strictly on the training set
train_temporal_X = generate_temporal_datasets(X_train_base)

# 4. Global Classifier Chain Training Loop
PENALTY_FACTOR = 5.0 

# Dictionary to store all trained models: all_predictors['admission']['FIBR_PREDS']
all_predictors = {} 

print("\nStarting Training Loop for All 4 Timelines (Admission, Day 1, Day 2, Day 3)...")
print("Warning: With time_limit=600 per target, full execution may take up to 8 hours.\n")

for time_stage, X_train_stage in train_temporal_X.items():
    print("="*60)
    print(f"--- TRAINING TIMELINE: {time_stage.upper()} ---")
    print("="*60)
    
    # Combine features and targets for AutoGluon
    train_data = pd.concat([X_train_stage, y_train_base], axis=1)
    train_data = TabularDataset(train_data)
    
    predictors_stage = {}
    predicted_train_features = train_data.copy()
    
    for i, target in enumerate(target_names):
        print(f"\nTraining Model for Target: {target} ({i+1}/{len(target_names)}) on {time_stage}...")
        
        future_targets = target_names[i+1:]
        train_input = predicted_train_features.drop(columns=future_targets).copy()
        
        # --- Intervention 1: Sample Weights ---
        if target in binary_targets:
            num_pos = (train_input[target] == 1).sum()
            num_neg = (train_input[target] == 0).sum()
            weight_ratio = (num_neg / (num_pos + 1e-5)) * PENALTY_FACTOR
            
            train_input['sample_weight'] = np.where(train_input[target] == 1, weight_ratio, 1.0)
            weight_col = 'sample_weight'
            metric = 'roc_auc'
            prob_type = 'binary'
        else:
            weight_col = None
            metric = 'accuracy'
            prob_type = 'multiclass'
            
        # --- Intervention 2: Optimized AutoGluon Training ---
        predictor = TabularPredictor(
            label=target, 
            eval_metric=metric,
            problem_type=prob_type,
            sample_weight=weight_col
        ).fit(
            train_input, 
            presets='best_quality', # Advanced Bagging/Stacking
            time_limit=300,         # More time for tree optimization
            excluded_model_types=['NN_TORCH', 'FASTAI'], # Exclude Deep Learning explicitly
            verbosity=0 
        )
        
        predictors_stage[target] = predictor
        
        # Remove weight column before predicting to feed the chain
        if weight_col:
            train_input = train_input.drop(columns=[weight_col])
            
        # Predict to append as a feature for the next target in the chain
        predicted_train_features[target] = predictor.predict(train_input)
        
    all_predictors[time_stage] = predictors_stage
    print(f"\nFinished training all 12 targets for timeline: {time_stage.upper()}")

print("\nAll timelines trained successfully. Models are stored in 'all_predictors'.")

Fetching dataset...


No path specified. Models will be saved in: "AutogluonModels/ag-20260323_164118"



Starting Training Loop for All 4 Timelines (Admission, Day 1, Day 2, Day 3)...

--- TRAINING TIMELINE: ADMISSION ---

Training Model for Target: FIBR_PREDS (1/12) on admission...


/usr/local/lib/python3.12/dist-packages/autogluon/tabular/predictor/predictor.py:1493: UserWarning: Failed to use ray for memory safe fits. Falling back to normal fit. Error: ValueError('ray==2.53.0 detected. 2.43.0 <= ray < 2.53.0 is required. You can use pip to install certain version of ray `pip install "ray>=2.43.0,<2.53.0"`')
  stacked_overfitting = self._sub_fit_memory_save_wrapper(
No path specified. Models will be saved in: "AutogluonModels/ag-20260323_164620"



Training Model for Target: PREDS_TAH (2/12) on admission...


...

/usr/local/lib/python3.12/dist-packages/autogluon/tabular/predictor/predictor.py:1493: UserWarning: Failed to use ray for memory safe fits. Falling back to normal fit. Error: ValueError('ray==2.53.0 detected. 2.43.0 <= ray < 2.53.0 is required. You can use pip to install certain version of ray `pip install "ray>=2.43.0,<2.53.0"`')
  stacked_overfitting = self._sub_fit_memory_save_wrapper(
No path specified. Models will be saved in: "AutogluonModels/ag-20260323_172135"



Training Model for Target: ZSN (9/12) on admission...


/usr/local/lib/python3.12/dist-packages/autogluon/tabular/predictor/predictor.py:1493: UserWarning: Failed to use ray for memory safe fits. Falling back to normal fit. Error: ValueError('ray==2.53.0 detected. 2.43.0 <= ray < 2.53.0 is required. You can use pip to install certain version of ray `pip install "ray>=2.43.0,<2.53.0"`')
  stacked_overfitting = self._sub_fit_memory_save_wrapper(
No path specified. Models will be saved in: "AutogluonModels/ag-20260323_172639"



Training Model for Target: REC_IM (10/12) on admission...


/usr/local/lib/python3.12/dist-packages/autogluon/tabular/predictor/predictor.py:1493: UserWarning: Failed to use ray for memory safe fits. Falling back to normal fit. Error: ValueError('ray==2.53.0 detected. 2.43.0 <= ray < 2.53.0 is required. You can use pip to install certain version of ray `pip install "ray>=2.43.0,<2.53.0"`')
  stacked_overfitting = self._sub_fit_memory_save_wrapper(
